# Advanced 08 lab — Optimize a multimodal inspection pipeline under edge constraints

**Scenario.** A small inspection service classifies product geometry, retrieves matching text descriptions, and must retain tiny-defect evidence. Site A fixes the reference. Site B is optimization-development only. Site C changes defect mix, request load, and repeated-asset rate and is reporting-only.

**Evidence boundary.** This credential-free notebook uses procedural 24×24 grayscale images, a tiny local dual encoder, and deterministic systems simulations. Timings are **CPU teaching measurements on this notebook host**—not target-device claims. Optional compilers and deployment runtimes are disabled. Candidate evidence never authorizes production.


## 1. Environment, reproducibility, and optional-runtime manifest

The default path uses common Python and PyTorch libraries. The optional-tool manifest is a governance record, not an import list. Hardware-specific performance requires a separate target environment.


In [ ]:
from __future__ import annotations

import copy
import hashlib
import json
import math
import platform
import random
import sys
import time
import tracemalloc
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score

SEED = 808
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = torch.device("cpu")
COURSE_DIR = Path.cwd()
if COURSE_DIR.name != "08-efficient-spatial-multimodal-inference":
    candidate = Path("curriculum/advanced/08-efficient-spatial-multimodal-inference")
    if candidate.exists():
        COURSE_DIR = candidate
ARTIFACT_DIR = COURSE_DIR / ".artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

RUN_OPTIONAL_COMPILE = False
OPTIONAL_SYSTEMS = pd.DataFrame([
    ("torch.compile", False, "host/compiler-dependent; coverage and parity required"),
    ("torchao", False, "target-kernel and quantization recipe required"),
    ("ExecuTorch", False, "target backend/device required"),
    ("ONNX Runtime", False, "isolated export/provider environment required"),
    ("TensorRT", False, "NVIDIA target required"),
    ("OpenVINO", False, "Intel target/plugin required"),
    ("Core ML", False, "Apple target and app integration required"),
    ("NVIDIA Triton", False, "separate serving/load-test environment required"),
], columns=["system", "enabled", "boundary"])

environment_manifest = {
    "python": sys.version.split()[0], "platform": platform.platform(),
    "torch": torch.__version__, "numpy": np.__version__, "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__, "device": str(DEVICE),
    "threads": torch.get_num_threads(), "seed": SEED,
    "benchmark_claim": "CPU teaching measurements on this notebook host only",
}
pd.DataFrame([environment_manifest]), OPTIONAL_SYSTEMS


## 2. Typed workload, metric, artifact, cache, and deployment contracts

Direction, slice, unit, aggregation, and required status travel with each metric. Missing evidence fails closed. The release record uses `PROMOTE_OPTIMIZED`, `KEEP_REFERENCE`, `REJECT`, or `MISSING_EVIDENCE`; its authorization is always `none`.


In [ ]:
@dataclass(frozen=True)
class SourceContract:
    site: str
    role: Literal["reference", "optimization_development", "reporting_only_no_changes"]
    revision: str

@dataclass(frozen=True)
class MetricSpec:
    name: str
    direction: Literal["higher_is_better", "lower_is_better"]
    unit: str
    required: bool = True

@dataclass(frozen=True)
class WorkloadContract:
    input_shape: tuple[int, int, int]
    service_mode: str
    batch_sizes: tuple[int, ...]
    warmup_runs: int
    measured_runs: int
    timed_inner_loops: int
    included_stages: tuple[str, ...]
    benchmark_claim: str

@dataclass(frozen=True)
class ArtifactContract:
    artifact_id: str
    parent_digest: str
    transformations: tuple[str, ...]
    precision: str
    shape_policy: str
    runtime: str
    rollback_artifact: str

@dataclass(frozen=True)
class GateCheck:
    name: str
    status: Literal["PASS", "FAIL", "MISSING"]
    detail: str

@dataclass(frozen=True)
class DeploymentDecision:
    candidate: str
    outcome: Literal["PROMOTE_OPTIMIZED", "KEEP_REFERENCE", "REJECT", "MISSING_EVIDENCE"]
    checks: tuple[GateCheck, ...]
    policy_hash: str
    authorization: Literal["none"] = "none"

SOURCES = (
    SourceContract("Site A", "reference", "camera-a-v1"),
    SourceContract("Site B", "optimization_development", "camera-b-v1"),
    SourceContract("Site C", "reporting_only_no_changes", "camera-c-v1"),
)
WORKLOAD = WorkloadContract(
    input_shape=(1, 24, 24), service_mode="interactive_cpu_teaching_proxy",
    batch_sizes=(1, 2, 4, 8), warmup_runs=5, measured_runs=30, timed_inner_loops=100,
    included_stages=("preprocess", "vision_encoder", "task_heads", "retrieval", "postprocess"),
    benchmark_claim="Demonstration thresholds and CPU measurements for this notebook runtime only",
)
METRICS = {
    "accuracy": MetricSpec("accuracy", "higher_is_better", "fraction"),
    "small_defect_recall": MetricSpec("small_defect_recall", "higher_is_better", "fraction"),
    "retrieval_recall": MetricSpec("retrieval_recall", "higher_is_better", "fraction"),
    "ece": MetricSpec("ece", "lower_is_better", "fraction"),
    "deterministic_service_p95_ms": MetricSpec("deterministic_service_p95_ms", "lower_is_better", "milliseconds"),
    "peak_memory_mb": MetricSpec("peak_memory_mb", "lower_is_better", "megabytes"),
}

def canonical_hash(payload) -> str:
    return hashlib.sha256(json.dumps(payload, sort_keys=True, separators=(",", ":"), default=str).encode()).hexdigest()

def state_digest(model: nn.Module) -> str:
    digest = hashlib.sha256()
    for name, tensor in sorted(model.state_dict().items()):
        digest.update(name.encode())
        digest.update(tensor.detach().cpu().numpy().tobytes())
    return digest.hexdigest()

assert all(spec.direction in {"higher_is_better", "lower_is_better"} for spec in METRICS.values())
pd.DataFrame([asdict(source) for source in SOURCES]), pd.DataFrame([asdict(metric) for metric in METRICS.values()])


## 3. Procedural Site A/B/C workload

Three geometric classes carry different global structure. A subset contains a one-pixel critical defect in a background region. Site B adds mild noise and contrast change. Site C has stronger shift, more tiny defects, higher request concurrency, and a different repeated-asset rate. Site C is created now but never enters optimization selection.


In [ ]:
CLASS_NAMES = ("vertical seal", "horizontal seal", "diagonal seal")
TEXT_DESCRIPTIONS = tuple(f"inspection image with {name}" for name in CLASS_NAMES)

def make_image(label: int, defect: bool, site: str, rng: np.random.Generator):
    image = np.full((24, 24), 0.08, dtype=np.float32)
    if label == 0:
        image[4:20, 10:14] = 0.62
    elif label == 1:
        image[10:14, 4:20] = 0.62
    else:
        for row in range(4, 20):
            image[row, row] = 0.62
            image[row, min(row + 1, 23)] = 0.62
    defect_xy = (-1, -1)
    if defect:
        candidates = [(3, 19), (19, 3), (20, 18), (4, 5)]
        y, x = candidates[int(rng.integers(0, len(candidates)))]
        image[y, x] = 0.88
        defect_xy = (y, x)
    if site == "Site B":
        image = image * 0.92 + 0.035 + rng.normal(0, 0.012, image.shape)
    elif site == "Site C":
        image = image * 0.82 + 0.075 + rng.normal(0, 0.025, image.shape)
        image = np.roll(image, 1, axis=1)
        if defect_xy != (-1, -1):
            defect_xy = (defect_xy[0], (defect_xy[1] + 1) % 24)
    return np.clip(image, 0, 1).astype(np.float32), defect_xy

def make_site(site: str, role: str, n: int, defect_rate: float, seed: int):
    rng = np.random.default_rng(seed)
    rows, images, labels, defects, points = [], [], [], [], []
    for index in range(n):
        label = index % len(CLASS_NAMES)
        defect = bool(rng.random() < defect_rate)
        image, point = make_image(label, defect, site, rng)
        asset_group = f"{site.lower().replace(' ', '-')}-{index // 3:03d}"
        rows.append({"asset_id": f"{site}-{index:03d}", "asset_group": asset_group, "site": site, "role": role})
        images.append(image); labels.append(label); defects.append(int(defect)); points.append(point)
    return {
        "x": torch.tensor(np.stack(images))[:, None],
        "y": torch.tensor(labels, dtype=torch.long),
        "defect": torch.tensor(defects, dtype=torch.float32),
        "defect_xy": points, "frame": pd.DataFrame(rows),
    }

site_a = make_site("Site A", "reference", 180, 0.35, SEED + 1)
site_b = make_site("Site B", "optimization_development", 120, 0.38, SEED + 2)
site_c = make_site("Site C", "reporting_only_no_changes", 120, 0.58, SEED + 3)
source_manifest = pd.DataFrame([
    {"site": name, "role": data["frame"]["role"].iloc[0], "rows": len(data["y"]), "defect_rate": float(data["defect"].mean())}
    for name, data in (("Site A", site_a), ("Site B", site_b), ("Site C", site_c))
])
source_manifest


## 4. Tiny multimodal reference model

The local model is deliberately small: two convolutional blocks, an image embedding, a class head, a defect head, and learned text embeddings in the same space. It is a mechanics proxy—not a foundation model or VLM benchmark.


In [ ]:
class TinyDualEncoder(nn.Module):
    def __init__(self, width: int = 12, embedding_dim: int = 8):
        super().__init__()
        hidden = max(4, width // 2)
        self.width = width
        self.features = nn.Sequential(
            nn.Conv2d(1, hidden, 3, padding=1), nn.ReLU(),
            nn.Conv2d(hidden, width, 3, padding=1), nn.ReLU(),
        )
        self.image_projection = nn.Linear(width * 9 + 1, embedding_dim)
        self.classifier = nn.Linear(embedding_dim, len(CLASS_NAMES))
        self.defect_head = nn.Linear(width * 9 + 1, 1)
        self.text_embeddings = nn.Parameter(torch.randn(len(CLASS_NAMES), embedding_dim) * 0.2)

    def forward(self, images):
        feature_map = self.features(images)
        pooled = F.adaptive_avg_pool2d(feature_map, (3, 3)).flatten(1)
        residual = (images - F.avg_pool2d(images, 3, stride=1, padding=1)).abs().amax(dim=(-2, -1))
        joined = torch.cat([pooled, residual], dim=1)
        image_embedding = F.normalize(self.image_projection(joined), dim=1)
        text_embedding = F.normalize(self.text_embeddings, dim=1)
        return {
            "logits": self.classifier(image_embedding),
            "defect_logit": self.defect_head(joined).squeeze(1),
            "image_embedding": image_embedding,
            "text_embedding": text_embedding,
        }

def train_model(width=12, epochs=55, distill_from=None, preserve_alignment=True):
    torch.manual_seed(SEED + width + epochs + int(preserve_alignment))
    model = TinyDualEncoder(width=width).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.025)
    images, labels, defects = site_a["x"].to(DEVICE), site_a["y"].to(DEVICE), site_a["defect"].to(DEVICE)
    teacher = distill_from
    if teacher is not None:
        teacher.eval()
        with torch.no_grad():
            teacher_out = teacher(images)
    history = []
    for epoch in range(epochs):
        model.train(); output = model(images)
        task_loss = F.cross_entropy(output["logits"], labels)
        defect_loss = F.binary_cross_entropy_with_logits(output["defect_logit"], defects)
        alignment_loss = F.cross_entropy(output["image_embedding"] @ output["text_embedding"].T / 0.12, labels)
        loss = task_loss + 0.45 * defect_loss + (0.55 if preserve_alignment else 0.0) * alignment_loss
        if teacher is not None:
            logit_loss = F.kl_div(
                F.log_softmax(output["logits"] / 2.0, dim=1),
                F.softmax(teacher_out["logits"] / 2.0, dim=1), reduction="batchmean"
            ) * 4.0
            loss = loss + 0.35 * logit_loss
            if preserve_alignment:
                loss = loss + 0.25 * F.mse_loss(output["image_embedding"], teacher_out["image_embedding"])
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        if epoch in {0, epochs - 1}:
            history.append({"epoch": epoch + 1, "loss": float(loss.detach())})
    return model.eval(), pd.DataFrame(history)

reference_model, reference_history = train_model()
REFERENCE_DIGEST = state_digest(reference_model)
reference_contract = ArtifactContract(
    "reference-fp32", REFERENCE_DIGEST, ("trained_site_a",), "fp32", "bounded_dynamic_batch_and_resolution", "pytorch_eager_cpu", "reference-fp32"
)
sum(p.numel() for p in reference_model.parameters()), reference_history, asdict(reference_contract)


## 5. Capability metrics and Site-B-only threshold selection

Calibration uses ECE with explicit bins. Retrieval is reported in both directions. The defect threshold is selected on Site B and then frozen. Site C remains untouched.


In [ ]:
def resize_images(images, resolution):
    if resolution == images.shape[-1]:
        return images
    return F.interpolate(images, size=(resolution, resolution), mode="bilinear", align_corners=False)

def expected_calibration_error(probabilities, labels, bins=8):
    confidence, prediction = probabilities.max(dim=1)
    edges = torch.linspace(0, 1, bins + 1)
    ece = torch.tensor(0.0)
    for lower, upper in zip(edges[:-1], edges[1:]):
        mask = (confidence > lower) & (confidence <= upper)
        if mask.any():
            ece += mask.float().mean() * (prediction[mask].eq(labels[mask]).float().mean() - confidence[mask].mean()).abs()
    return float(ece)

def select_defect_threshold(model, dataset):
    with torch.inference_mode():
        scores = torch.sigmoid(model(dataset["x"])["defect_logit"]).numpy()
    truth = dataset["defect"].numpy().astype(int)
    candidates = np.linspace(0.05, 0.95, 91)
    rows = []
    for threshold in candidates:
        pred = scores >= threshold
        tp = int(((pred == 1) & (truth == 1)).sum()); fn = int(((pred == 0) & (truth == 1)).sum())
        fp = int(((pred == 1) & (truth == 0)).sum())
        recall = tp / max(tp + fn, 1); precision = tp / max(tp + fp, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-12)
        rows.append((threshold, recall, precision, f1))
    return max(rows, key=lambda row: (row[3], row[1]))[0], pd.DataFrame(rows, columns=["threshold", "recall", "precision", "f1"])

DEFECT_THRESHOLD, defect_threshold_sweep = select_defect_threshold(reference_model, site_b)

def evaluate_capabilities(model, dataset, resolution=24, defect_threshold=DEFECT_THRESHOLD, logit_scale=1.0):
    images = resize_images(dataset["x"], resolution)
    with torch.inference_mode():
        output = model(images)
    logits = output["logits"] * logit_scale
    probs = logits.softmax(dim=1)
    labels = dataset["y"]
    predictions = probs.argmax(1)
    defect_prediction = torch.sigmoid(output["defect_logit"]) >= defect_threshold
    positives = dataset["defect"] == 1
    small_recall = float(defect_prediction[positives].float().mean()) if positives.any() else float("nan")
    image_to_text = (output["image_embedding"] @ output["text_embedding"].T).argmax(1)
    image_to_text_recall = float(image_to_text.eq(labels).float().mean())
    similarities = output["text_embedding"] @ output["image_embedding"].T
    text_to_image_hits = []
    for class_index in range(len(CLASS_NAMES)):
        top5 = similarities[class_index].topk(5).indices
        text_to_image_hits.append(float(labels[top5].eq(class_index).any()))
    return {
        "accuracy": accuracy_score(labels.numpy(), predictions.numpy()),
        "macro_f1": f1_score(labels.numpy(), predictions.numpy(), average="macro"),
        "small_defect_recall": small_recall,
        "image_to_text_recall": image_to_text_recall,
        "text_to_image_recall_at_5": float(np.mean(text_to_image_hits)),
        "retrieval_recall": min(image_to_text_recall, float(np.mean(text_to_image_hits))),
        "ece": expected_calibration_error(probs, labels),
    }

reference_capability = pd.DataFrame([
    {"site": name, **evaluate_capabilities(reference_model, data)}
    for name, data in (("Site A", site_a), ("Site B", site_b))
])
assert "Site C" not in set(reference_capability["site"])
reference_capability, DEFECT_THRESHOLD


## 6. End-to-end stage profiler, cold start, and steady state

The first request is reported separately. Stage timing includes an explicit preprocessing copy, encoder/heads, retrieval, and postprocessing/policy proxy. Python timing noise is visible; these values support the lesson, not a hardware claim.


In [ ]:
def percentile(values, q):
    return float(np.quantile(np.asarray(values, dtype=float), q))

def profile_pipeline(model, sample, repetitions=30):
    stage_rows, total = [], []
    model.eval()
    for repetition in range(repetitions):
        t0 = time.perf_counter_ns()
        image = sample.detach().clone().contiguous()
        t1 = time.perf_counter_ns()
        with torch.inference_mode():
            output = model(image)
        t2 = time.perf_counter_ns()
        _ = output["image_embedding"] @ output["text_embedding"].T
        t3 = time.perf_counter_ns()
        _ = int(output["logits"].argmax(1)[0])
        t4 = time.perf_counter_ns()
        ms = np.diff([t0, t1, t2, t3, t4]) / 1e6
        stage_rows.append(dict(zip(("preprocess", "vision_encoder_and_heads", "retrieval", "postprocess_policy"), ms)))
        total.append((t4 - t0) / 1e6)
    return pd.DataFrame(stage_rows), np.asarray(total)

sample = site_b["x"][:1]
cold_start_begin = time.perf_counter_ns()
with torch.inference_mode():
    _ = reference_model(sample)
cold_start_ms = (time.perf_counter_ns() - cold_start_begin) / 1e6
for _ in range(WORKLOAD.warmup_runs):
    with torch.inference_mode(): _ = reference_model(sample)
stage_times, steady_times = profile_pipeline(reference_model, sample, WORKLOAD.measured_runs)
profile_summary = pd.DataFrame([
    {"stage": column, "p50_ms": percentile(stage_times[column], .50), "p95_ms": percentile(stage_times[column], .95),
     "iqr_ms": percentile(stage_times[column], .75) - percentile(stage_times[column], .25),
     "mean_ms": float(stage_times[column].mean()), "repetitions": WORKLOAD.measured_runs}
    for column in stage_times
])
system_baseline = {
    "cold_start_ms": cold_start_ms, "steady_p50_ms": percentile(steady_times, .50),
    "steady_p95_ms": percentile(steady_times, .95), "steady_p99_ms": percentile(steady_times, .99),
    "steady_iqr_ms": percentile(steady_times, .75) - percentile(steady_times, .25),
    "max_ms": float(steady_times.max()), "repetitions": WORKLOAD.measured_runs,
}
pd.DataFrame([system_baseline]), profile_summary


## 7. Parameter, artifact, and peak-memory evidence

Artifact bytes, parameter bytes, and process peak are different. `tracemalloc` sees Python allocations, not every native runtime allocation, so the notebook labels it as a proxy and also reports tensor-based estimates.


In [ ]:
def model_footprint(model, bytes_per_parameter=4):
    parameters = sum(p.numel() for p in model.parameters())
    return {
        "parameters": parameters,
        "parameter_memory_mb": parameters * bytes_per_parameter / 1e6,
        "artifact_size_mb_proxy": parameters * bytes_per_parameter / 1e6,
    }

tracemalloc.start()
with torch.inference_mode(): _ = reference_model(site_b["x"][:8])
_, traced_peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
reference_footprint = model_footprint(reference_model)
reference_footprint["python_traced_peak_mb_proxy"] = traced_peak / 1e6
reference_footprint["input_tensor_mb_b8"] = site_b["x"][:8].numel() * site_b["x"].element_size() / 1e6
pd.DataFrame([reference_footprint])


## 8. Resolution sweep: aggregate quality can hide a tiny-evidence collapse

The model accepts bounded dynamic spatial shapes. No upsampling hides the compute change. Each host benchmark repetition times 100 forward calls as one region, normalizes per batch, and reports median, p95, IQR, repetition count, and timed-region duration. This reduces timer granularity error but remains host-specific diagnostic evidence—not the deterministic service model used by release gates. We report input bytes, timing dispersion, classification, retrieval, calibration, and the critical small-defect slice together.


In [ ]:
def benchmark_forward(model, images, repetitions=25, inner_loops=WORKLOAD.timed_inner_loops):
    for _ in range(4):
        with torch.inference_mode(): _ = model(images)
    normalized_times, timed_regions = [], []
    for _ in range(repetitions):
        start = time.perf_counter_ns()
        with torch.inference_mode():
            for _ in range(inner_loops):
                _ = model(images)
        region_ms = (time.perf_counter_ns() - start) / 1e6
        timed_regions.append(region_ms)
        normalized_times.append(region_ms / inner_loops)
    return {
        "host_p50_ms_per_batch": percentile(normalized_times, .50),
        "host_p95_ms_per_batch": percentile(normalized_times, .95),
        "host_mean_ms_per_batch": float(np.mean(normalized_times)),
        "host_iqr_ms_per_batch": percentile(normalized_times, .75) - percentile(normalized_times, .25),
        "benchmark_repetitions": repetitions,
        "timed_inner_loops": inner_loops,
        "timed_region_p50_ms": percentile(timed_regions, .50),
    }

resolution_rows = []
for resolution in (24, 16, 12, 8):
    resized = resize_images(site_b["x"], resolution)
    metrics = evaluate_capabilities(reference_model, site_b, resolution=resolution)
    timing = benchmark_forward(reference_model, resized[:1])
    resolution_rows.append({
        "resolution": resolution, "input_bytes": resized[:1].numel() * resized.element_size(),
        **timing, **metrics,
    })
resolution_results = pd.DataFrame(resolution_rows)
assert resolution_results.loc[resolution_results["resolution"] == 8, "input_bytes"].iloc[0] < resolution_results.loc[resolution_results["resolution"] == 24, "input_bytes"].iloc[0]
resolution_results


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(resolution_results["resolution"], resolution_results["accuracy"], "o-", label="accuracy")
axes[0].plot(resolution_results["resolution"], resolution_results["small_defect_recall"], "o-", label="small-defect recall")
axes[0].set(xlabel="input resolution", ylabel="quality", title="Quality is slice-dependent"); axes[0].legend()
axes[1].plot(resolution_results["resolution"], resolution_results["host_p95_ms_per_batch"], "o-", color="#F59E42")
axes[1].set(xlabel="input resolution", ylabel="CPU p95 ms / batch", title="Measured host timing (100 calls / region)")
plt.tight_layout(); plt.show()


## 9. Dynamic-resolution cascade and router error

Site B chooses a confidence gate. Low-resolution predictions below the threshold escalate to 24×24. We measure the false low-resolution acceptance rate explicitly; confidence alone is not evidence that a small defect survived.


In [ ]:
def cascade_report(model, dataset, low_resolution=12, confidence_threshold=0.86):
    with torch.inference_mode():
        low = model(resize_images(dataset["x"], low_resolution))
        high = model(dataset["x"])
    low_prob = low["logits"].softmax(1); high_prob = high["logits"].softmax(1)
    low_conf, low_pred = low_prob.max(1); high_pred = high_prob.argmax(1)
    escalate = low_conf < confidence_threshold
    final_pred = torch.where(escalate, high_pred, low_pred)
    low_defect_pred = torch.sigmoid(low["defect_logit"]) >= DEFECT_THRESHOLD
    critical_failure = (dataset["defect"] == 1) & (~low_defect_pred)
    false_low_accept = (~escalate) & (low_pred.ne(dataset["y"]) | critical_failure)
    low_cost = low_resolution ** 2; high_cost = 24 ** 2
    return {
        "confidence_threshold": confidence_threshold,
        "escalation_rate": float(escalate.float().mean()),
        "false_low_resolution_acceptance_rate": float(false_low_accept.float().mean()),
        "accuracy": float(final_pred.eq(dataset["y"]).float().mean()),
        "low_resolution_small_defect_recall": float(low_defect_pred[dataset["defect"] == 1].float().mean()),
        "relative_pixel_compute": float((low_cost + escalate.float().mean() * high_cost) / high_cost),
    }

cascade_sweep = pd.DataFrame([cascade_report(reference_model, site_b, confidence_threshold=t) for t in (0.65, 0.75, 0.85, 0.92)])
CASCADE_THRESHOLD = float(cascade_sweep.sort_values(["false_low_resolution_acceptance_rate", "relative_pixel_compute"]).iloc[0]["confidence_threshold"])
cascade_sweep


## 10. Visual-token budget and critical-evidence retention

Patches are scored by variance as a transparent salience proxy. This is not a production pruning method. A known critical patch lets us test whether pruning preserved required evidence rather than only whether the class token stayed stable.


In [ ]:
def patch_salience_and_critical(dataset, patch=4):
    images = dataset["x"]
    unfolded = images.unfold(2, patch, patch).unfold(3, patch, patch)
    # Global-class salience proxy: patch mean favors the large seal and can miss a tiny defect.
    salience = unfolded.mean(dim=(-1, -2)).squeeze(1)
    critical = []
    for point in dataset["defect_xy"]:
        if point == (-1, -1):
            critical.append(-1)
        else:
            y, x = point
            critical.append((y // patch) * (24 // patch) + (x // patch))
    salience = salience.flatten(1)
    # This class-only proxy has no critical-defect supervision; make that failure surface explicit.
    for row, token in enumerate(critical):
        if token >= 0:
            salience[row, token] *= 0.05
    return salience, np.asarray(critical)

salience, critical_token = patch_salience_and_critical(site_b)
token_rows = []
for retention in (1.0, .75, .50, .25):
    keep = max(1, int(salience.shape[1] * retention))
    retained = salience.topk(keep, dim=1).indices.numpy()
    positive_rows = np.where(critical_token >= 0)[0]
    evidence = [critical_token[row] in retained[row] for row in positive_rows]
    token_rows.append({
        "token_retention": retention, "tokens_kept": keep,
        "attention_interaction_proxy": keep ** 2,
        "small_evidence_retention_rate": float(np.mean(evidence)),
    })
token_budget_results = pd.DataFrame(token_rows)

# Known-answer signature failure: a low-salience but required token is pruned.
toy_salience = torch.tensor([0.9, 0.8, 0.7, 0.01, 0.6, 0.5])
toy_critical_token = 3
toy_retained = toy_salience.topk(3).indices.tolist()
assert toy_critical_token not in toy_retained
token_budget_results, {"toy_critical_token": toy_critical_token, "retained": toy_retained, "failure": "required evidence pruned"}


## 11. INT8-like PTQ proxy and a same-accuracy calibration failure

Weights are quantized and dequantized per tensor using Site-B-governed configuration. The unsafe candidate also carries an intentionally injected output-scale mismatch so learners can see that unchanged argmax accuracy does not imply unchanged confidence. This is a transparent failure injection, not a claim that all INT8 runtimes behave this way.


In [ ]:
def quantize_dequantize_tensor(tensor, levels=255):
    minimum, maximum = tensor.min(), tensor.max()
    if float(maximum - minimum) == 0:
        return tensor.clone()
    scale = (maximum - minimum) / levels
    quantized = torch.round((tensor - minimum) / scale).clamp(0, levels)
    return quantized * scale + minimum

def ptq_proxy(model):
    candidate = copy.deepcopy(model)
    with torch.no_grad():
        for parameter in candidate.parameters():
            parameter.copy_(quantize_dequantize_tensor(parameter))
    return candidate.eval()

quantized_model = ptq_proxy(reference_model)
quantized_metrics = evaluate_capabilities(quantized_model, site_b)
quantized_calibration_failure = evaluate_capabilities(quantized_model, site_b, logit_scale=0.15)
ptq_comparison = pd.DataFrame([
    {"candidate": "reference_fp32", **evaluate_capabilities(reference_model, site_b), **model_footprint(reference_model, 4)},
    {"candidate": "int8_like_weights", **quantized_metrics, **model_footprint(quantized_model, 1)},
    {"candidate": "int8_like_with_output_scale_mismatch", **quantized_calibration_failure, **model_footprint(quantized_model, 1)},
])
assert ptq_comparison.loc[0, "accuracy"] == ptq_comparison.loc[2, "accuracy"]
ptq_comparison[["candidate", "accuracy", "small_defect_recall", "retrieval_recall", "ece", "artifact_size_mb_proxy"]]


## 12. Structured channel-reduction curve

The CPU-safe proxy creates genuinely narrower convolution tensors, not a dense model with zeros. Each width is retrained under one fixed Site-A recipe and evaluated on Site B. A production pruning study would inherit weights, fine-tune, repeat seeds, and benchmark supported kernels on target hardware.


In [ ]:
pruned_models = {12: reference_model}
pruning_rows = []
for width in (12, 9, 6, 4):
    model = reference_model if width == 12 else train_model(width=width, epochs=38)[0]
    pruned_models[width] = model
    metrics = evaluate_capabilities(model, site_b)
    footprint = model_footprint(model)
    timing = benchmark_forward(model, site_b["x"][:1], repetitions=20)
    pruning_rows.append({
        "width": width, "structured_pruning_fraction": 1 - width / 12,
        **footprint, **timing, **metrics,
    })
pruning_results = pd.DataFrame(pruning_rows)
assert pruning_results.loc[pruning_results["width"] == 4, "parameters"].iloc[0] < pruning_results.loc[pruning_results["width"] == 12, "parameters"].iloc[0]
pruning_results[["width", "structured_pruning_fraction", "parameters", "host_p95_ms_per_batch", "host_iqr_ms_per_batch", "accuracy", "small_defect_recall", "retrieval_recall", "ece"]]


## 13. Task-only versus multimodal distillation

Both students see teacher logits. Only the multimodal student also preserves alignment and representation geometry. Classification agreement is not the acceptance criterion.


In [ ]:
task_only_student, _ = train_model(width=6, epochs=45, distill_from=reference_model, preserve_alignment=False)
multimodal_student, _ = train_model(width=6, epochs=45, distill_from=reference_model, preserve_alignment=True)
distillation_results = pd.DataFrame([
    {"candidate": "teacher", **model_footprint(reference_model), **benchmark_forward(reference_model, site_b["x"][:1], 20), **evaluate_capabilities(reference_model, site_b)},
    {"candidate": "task_only_student", **model_footprint(task_only_student), **benchmark_forward(task_only_student, site_b["x"][:1], 20), **evaluate_capabilities(task_only_student, site_b)},
    {"candidate": "multimodal_student", **model_footprint(multimodal_student), **benchmark_forward(multimodal_student, site_b["x"][:1], 20), **evaluate_capabilities(multimodal_student, site_b)},
])
distillation_results[["candidate", "parameters", "host_p95_ms_per_batch", "host_iqr_ms_per_batch", "accuracy", "small_defect_recall", "image_to_text_recall", "text_to_image_recall_at_5", "ece"]]


## 14. Export/compilation contract, graph coverage, and numeric parity

`torch.export` is attempted because it is available in the tested PyTorch environment. `torch.compile` stays disabled by default because first-run cost and backend support vary across CI and learner hosts. Export success is not a speed claim. Numeric parity and behavioral parity are separate: the cell includes a near-boundary counterexample where a sub-micro numeric change flips the predicted class.


In [ ]:
class ExportableReference(nn.Module):
    def __init__(self, model):
        super().__init__(); self.model = model
    def forward(self, images):
        output = self.model(images)
        return output["logits"], output["defect_logit"], output["image_embedding"]

export_module = ExportableReference(reference_model).eval()
export_status = {"exported": False, "graph_breaks_allowed": False, "error": None}
try:
    exported_program = torch.export.export(export_module, (site_b["x"][:1],))
    with torch.inference_mode():
        eager_outputs = export_module(site_b["x"][:1])
        exported_outputs = exported_program.module()(site_b["x"][:1])
    differences = torch.cat([(left - right).abs().flatten() for left, right in zip(eager_outputs, exported_outputs)])
    task_disagreement = float(eager_outputs[0].argmax(1).ne(exported_outputs[0].argmax(1)).float().mean())
    export_status.update({
        "exported": True, "max_abs_difference": float(differences.max()),
        "mean_abs_difference": float(differences.mean()), "task_disagreement_rate": task_disagreement,
    })
except Exception as error:
    export_status["error"] = f"{type(error).__name__}: {error}"

# Tiny numeric error can still change behavior at a decision boundary.
boundary_source_logits = torch.tensor([[0.50000006, 0.50000000]], dtype=torch.float64)
boundary_runtime_logits = torch.tensor([[0.49999997, 0.50000009]], dtype=torch.float64)
boundary_abs_difference = float((boundary_source_logits - boundary_runtime_logits).abs().max())
boundary_behavioral_disagreement = bool(
    boundary_source_logits.argmax(1).ne(boundary_runtime_logits.argmax(1)).item()
)
assert boundary_abs_difference < 2e-7
assert boundary_behavioral_disagreement is True
behavioral_parity_counterexample = {
    "max_abs_difference": boundary_abs_difference,
    "source_class": int(boundary_source_logits.argmax(1).item()),
    "runtime_class": int(boundary_runtime_logits.argmax(1).item()),
    "behavioral_disagreement": boundary_behavioral_disagreement,
}

compile_status = {"enabled": RUN_OPTIONAL_COMPILE, "whole_graph_claim": False, "benchmark_claim": "none"}
if RUN_OPTIONAL_COMPILE:
    compiled_model = torch.compile(export_module, fullgraph=True)
    compile_status["benchmark_claim"] = benchmark_forward(compiled_model, site_b["x"][:1], 15)
export_contract = {
    "source_model_digest": REFERENCE_DIGEST, "export_format": "torch.export.ExportedProgram",
    "runtime_version": torch.__version__, "precision": "fp32",
    "dynamic_shape_policy": "static teaching export; production must declare bounded dynamics",
    "optimization_profile": "none", **export_status,
}
assert not export_status["exported"] or export_status["max_abs_difference"] <= 1e-6
export_contract, compile_status, behavioral_parity_counterexample


## 15. Batch sweep: throughput is not request latency

Batch measurements use actual forward passes on this host. Peak activation memory is a transparent tensor proxy, not process RSS.


In [ ]:
batch_rows = []
for batch in WORKLOAD.batch_sizes:
    images = site_b["x"][:batch]
    timing = benchmark_forward(reference_model, images, 25)
    batch_rows.append({
        "batch": batch, **timing,
        "per_request_host_p95_ms": timing["host_p95_ms_per_batch"] / batch,
        "throughput_requests_per_s": 1000 * batch / max(timing["host_mean_ms_per_batch"], 1e-9),
        "input_memory_mb_proxy": images.numel() * images.element_size() / 1e6,
    })
batch_results = pd.DataFrame(batch_rows)
batch_results


## 16. Dynamic batching, load, saturation, deadlines, and backpressure

The deterministic queue simulator includes arrival, wait-window, service, timeout, and bounded-queue behavior. It is a systems model—not a Triton benchmark.


In [ ]:
def simulate_dynamic_batching(arrival_rate_rps, duration_s=4.0, max_batch=4, wait_ms=3.0, queue_limit=None, timeout_ms=40.0, seed=SEED):
    rng = np.random.default_rng(seed + int(arrival_rate_rps) + int(wait_ms))
    arrivals = np.cumsum(rng.exponential(1000 / arrival_rate_rps, size=max(20, int(duration_s * arrival_rate_rps * 1.4))))
    arrivals = arrivals[arrivals <= duration_s * 1000]
    index, server_free, completed, rejected, timed_out, batches = 0, 0.0, [], 0, 0, []
    while index < len(arrivals):
        if server_free <= arrivals[index]:
            first = arrivals[index]
            kth = arrivals[min(index + max_batch - 1, len(arrivals) - 1)]
            ready_time = min(first + wait_ms, kth)
        else:
            ready_time = server_free
        available_end = max(index + 1, int(np.searchsorted(arrivals, ready_time, side="right")))
        queued = available_end - index
        if queue_limit is not None and queued > queue_limit:
            rejected_now = queued - queue_limit
            rejected += rejected_now
            index += rejected_now
            queued = queue_limit
        take = min(max_batch, queued)
        batch_arrivals = arrivals[index:index + take]
        service_ms = 2.2 + 1.25 * take
        service_start = max(server_free, ready_time)
        finish = service_start + service_ms
        for arrival in batch_arrivals:
            latency = finish - arrival
            if latency <= timeout_ms:
                completed.append((finish, latency, service_start - arrival, service_ms))
            else:
                timed_out += 1
        batches.append(take); server_free = finish; index += take
    latencies = np.asarray([row[1] for row in completed])
    queue = np.asarray([row[2] for row in completed])
    elapsed_s = max(server_free / 1000, duration_s)
    return {
        "arrival_rate_rps": arrival_rate_rps, "wait_ms": wait_ms,
        "mean_batch": float(np.mean(batches)) if batches else 0.0,
        "queue_p95_ms": percentile(queue, .95) if len(queue) else float("nan"),
        "end_to_end_p95_ms": percentile(latencies, .95) if len(latencies) else float("nan"),
        "throughput_rps": len(completed) / elapsed_s,
        "timeout_rate": timed_out / max(len(arrivals), 1),
        "reject_rate": rejected / max(len(arrivals), 1),
        "timeout_or_reject_rate": (timed_out + rejected) / max(len(arrivals), 1),
        "max_observed_batch": max(batches, default=0),
    }

load_results = pd.DataFrame([
    simulate_dynamic_batching(rate, wait_ms=wait, queue_limit=16)
    for rate in (40, 180, 420, 900) for wait in (0.0, 3.0, 8.0)
])
saturation_region = load_results[(load_results["timeout_or_reject_rate"] > 0) | (load_results["end_to_end_p95_ms"] > 30)]
load_results, saturation_region


## 17. Unbounded queue versus freshness-aware backpressure

For streaming perception, “completed eventually” is not necessarily useful. We compare a permissive queue with a bounded policy and preserve rejected work as rejected—not successful inference.


In [ ]:
backpressure_results = pd.DataFrame([
    {"policy": "unbounded_queue", **simulate_dynamic_batching(900, wait_ms=8, queue_limit=None, timeout_ms=250)},
    {"policy": "bounded_queue_reject_stale", **simulate_dynamic_batching(900, wait_ms=3, queue_limit=8, timeout_ms=40)},
])
backpressure_results["deadline_miss_or_reject_rate"] = backpressure_results["timeout_or_reject_rate"]
backpressure_results


## 18. Embedding cache and stale-cache attack

A filename-keyed cache returns the wrong embedding after the asset changes. A contract key binds content, encoder, processor, tenant, authenticated principal, and authorization scope and therefore misses safely. The same bytes and model cannot cross principals unless a separately reviewed artifact class is explicitly authorization-independent.


In [ ]:
def tensor_digest(tensor):
    return hashlib.sha256(tensor.detach().cpu().numpy().tobytes()).hexdigest()

def cache_key(image, encoder_digest, processor_revision, tenant, principal, authorization_scope):
    return canonical_hash({
        "content_digest": tensor_digest(image), "encoder_digest": encoder_digest,
        "processor_revision": processor_revision, "tenant": tenant,
        "principal": principal,
        "authorization_scope": authorization_scope,
    })

asset_v1 = site_b["x"][0].clone()
asset_v2 = asset_v1.clone(); asset_v2[:, 2, 2] = 1.0
with torch.inference_mode():
    embedding_v1 = reference_model(asset_v1[None])["image_embedding"][0]

unsafe_filename_cache = {"inspection.png": embedding_v1}
unsafe_stale_hit = "inspection.png" in unsafe_filename_cache
safe_cache = {cache_key(asset_v1, REFERENCE_DIGEST, "processor-v1", "tenant-a", "alice", "inspect"): embedding_v1}
v2_key = cache_key(asset_v2, REFERENCE_DIGEST, "processor-v1", "tenant-a", "alice", "inspect")
safe_v2_hit = v2_key in safe_cache
wrong_tenant_key = cache_key(asset_v1, REFERENCE_DIGEST, "processor-v1", "tenant-b", "alice", "inspect")
wrong_principal_key = cache_key(asset_v1, REFERENCE_DIGEST, "processor-v1", "tenant-a", "bob", "inspect")
assert unsafe_stale_hit is True
assert safe_v2_hit is False
assert wrong_tenant_key not in safe_cache
assert wrong_principal_key not in safe_cache
cache_attack_results = pd.DataFrame([
    {"cache": "filename_only", "asset_changed": True, "hit": unsafe_stale_hit, "valid_hit": False, "outcome": "STALE_HIT"},
    {"cache": "contract_digest", "asset_changed": True, "hit": safe_v2_hit, "valid_hit": False, "outcome": "SAFE_MISS"},
    {"cache": "contract_digest_wrong_tenant", "asset_changed": False, "hit": wrong_tenant_key in safe_cache, "valid_hit": False, "outcome": "SAFE_MISS"},
    {"cache": "contract_digest_wrong_principal", "asset_changed": False, "hit": wrong_principal_key in safe_cache, "valid_hit": False, "outcome": "SAFE_MISS"},
])
cache_attack_results


## 19. Temporal reuse and adaptive video sampling

The synthetic motion/event signal compares full inference, periodic reuse, and adaptive sampling. Compute saved is reported next to event recall and worst state age.


In [ ]:
frames = np.arange(40)
event = np.isin(frames, [7, 8, 23, 24])
motion = np.where(event, 0.9, 0.08) + np.random.default_rng(SEED).normal(0, 0.02, len(frames))

def sampling_report(name, selected):
    selected = np.asarray(sorted(set(selected)), dtype=int)
    observed_event_frames = set(selected[event[selected]])
    event_groups = [set([7, 8]), set([23, 24])]
    event_recall = np.mean([bool(group & observed_event_frames) for group in event_groups])
    state_age, last = [], 0
    for frame in frames:
        if frame in set(selected): last = frame
        state_age.append(frame - last)
    return {
        "policy": name, "full_inferences": len(selected),
        "compute_proxy_relative": len(selected) / len(frames),
        "event_recall": event_recall, "max_state_age_frames": max(state_age),
    }

temporal_reuse_results = pd.DataFrame([
    sampling_report("full_inference", frames),
    sampling_report("periodic_every_5", frames[::5]),
    sampling_report("adaptive_motion", np.where((motion > 0.35) | (frames % 8 == 0))[0]),
])
temporal_reuse_results


## 20. Candidate matrix

Every row carries systems and capability evidence. Host timing is measured with long timed regions and dispersion, but release comparisons use a deterministic service-cost model so scheduler jitter cannot alter the lesson. Token pruning remains a policy proxy, so its token retention and small-evidence retention are explicit rather than silently folded into classification.


In [ ]:
REFERENCE_PARAMETERS = sum(parameter.numel() for parameter in reference_model.parameters())

def deterministic_service_latency(model, resolution=24, bytes_per_parameter=4, token_retention=1.0, workload_multiplier=1.0):
    parameter_ratio = sum(parameter.numel() for parameter in model.parameters()) / REFERENCE_PARAMETERS
    resolution_ratio = (resolution / 24) ** 2
    precision_execution_factor = 0.68 if bytes_per_parameter == 1 else 1.0
    p50_ms = workload_multiplier * (
        1.5
        + 3.0 * resolution_ratio * token_retention
        + 3.5 * parameter_ratio * precision_execution_factor
    )
    return {
        "deterministic_service_p50_ms": p50_ms,
        "deterministic_service_p95_ms": p50_ms * 1.12,
        "service_latency_source": "deterministic_workload_model_v1",
    }

def candidate_row(name, model, resolution=24, bytes_per_parameter=4, logit_scale=1.0, evidence_retention=1.0, token_retention=1.0, cache_multiplier=1.0):
    capabilities = evaluate_capabilities(model, site_b, resolution=resolution, logit_scale=logit_scale)
    timing = benchmark_forward(model, resize_images(site_b["x"][:1], resolution), 20)
    service_timing = deterministic_service_latency(model, resolution, bytes_per_parameter, token_retention)
    footprint = model_footprint(model, bytes_per_parameter)
    return {
        "candidate": name, **timing, **service_timing,
        "peak_memory_mb": footprint["parameter_memory_mb"] + resize_images(site_b["x"][:8], resolution).numel() * 4 / 1e6,
        "artifact_size_mb": footprint["artifact_size_mb_proxy"],
        "token_retention_rate": token_retention,
        "small_evidence_retention_rate": evidence_retention,
        "cache_latency_multiplier": cache_multiplier,
        **capabilities,
    }

candidate_rows = [
    candidate_row("reference", reference_model),
    candidate_row("low_resolution_fast_unsafe", reference_model, resolution=8),
    candidate_row("token_pruning_50", reference_model, token_retention=0.5, evidence_retention=float(token_budget_results.query("token_retention == 0.5")["small_evidence_retention_rate"].iloc[0])),
    candidate_row("int8_like", quantized_model, bytes_per_parameter=1),
    candidate_row("int8_same_accuracy_bad_calibration", quantized_model, bytes_per_parameter=1, logit_scale=0.15),
    candidate_row("structured_width_6", pruned_models[6]),
    candidate_row("task_only_student", task_only_student),
    candidate_row("multimodal_student", multimodal_student),
]
candidate_matrix = pd.DataFrame(candidate_rows)
candidate_matrix


## 21. Feasibility gate and explicit `MISSING`

Hard capability, reliability, evidence, and resource constraints are evaluated **before** operational Pareto selection. Timing gates consume the deterministic service model, not sub-millisecond host measurements. These are demonstration thresholds for this synthetic workload only; they must not be copied to another service or device.


In [ ]:
site_b_reference = candidate_matrix.set_index("candidate").loc["reference"]
DEPLOYMENT_BUDGET = {
    "notice": "Demonstration thresholds for this deterministic synthetic workload only",
    "deterministic_service_p95_ms_max": float(site_b_reference["deterministic_service_p95_ms"] * 1.20),
    "deterministic_end_to_end_p95_ms_max": 25.0,
    "peak_memory_mb_max": float(site_b_reference["peak_memory_mb"] * 1.01),
    "artifact_size_mb_max": float(site_b_reference["artifact_size_mb"] * 1.01),
    "accuracy_min": float(site_b_reference["accuracy"] - 0.03),
    "small_defect_recall_min": float(max(0, site_b_reference["small_defect_recall"] - 0.08)),
    "retrieval_recall_min": float(max(0, site_b_reference["retrieval_recall"] - 0.08)),
    "ece_max": float(site_b_reference["ece"] + 0.04),
    "small_evidence_retention_rate_min": 0.90,
    "queue_timeout_rate_max": 0.05,
}

def gate_value(row, key, threshold, relation, label):
    if key not in row or pd.isna(row[key]):
        return GateCheck(label, "MISSING", f"required `{key}` missing")
    passed = row[key] <= threshold if relation == "max" else row[key] >= threshold
    return GateCheck(label, "PASS" if passed else "FAIL", f"observed={row[key]:.6g}; {relation}={threshold:.6g}")

def evaluate_gate(row):
    return (
        gate_value(row, "deterministic_service_p95_ms", DEPLOYMENT_BUDGET["deterministic_service_p95_ms_max"], "max", "service_latency"),
        gate_value(row, "peak_memory_mb", DEPLOYMENT_BUDGET["peak_memory_mb_max"], "max", "peak_memory"),
        gate_value(row, "artifact_size_mb", DEPLOYMENT_BUDGET["artifact_size_mb_max"], "max", "artifact_size"),
        gate_value(row, "accuracy", DEPLOYMENT_BUDGET["accuracy_min"], "min", "classification"),
        gate_value(row, "small_defect_recall", DEPLOYMENT_BUDGET["small_defect_recall_min"], "min", "small_defect"),
        gate_value(row, "retrieval_recall", DEPLOYMENT_BUDGET["retrieval_recall_min"], "min", "retrieval"),
        gate_value(row, "ece", DEPLOYMENT_BUDGET["ece_max"], "max", "calibration"),
        gate_value(row, "small_evidence_retention_rate", DEPLOYMENT_BUDGET["small_evidence_retention_rate_min"], "min", "critical_evidence"),
    )

gate_rows = []
for _, row in candidate_matrix.iterrows():
    checks = evaluate_gate(row)
    statuses = [check.status for check in checks]
    gate_rows.append({"candidate": row["candidate"], "eligible": all(status == "PASS" for status in statuses), "statuses": {check.name: check.status for check in checks}})
gate_results = pd.DataFrame(gate_rows)

missing_example = candidate_matrix.iloc[0].drop(labels=["small_defect_recall"])
assert {check.name: check.status for check in evaluate_gate(missing_example)}["small_defect"] == "MISSING"
fast_unsafe_status = gate_results.set_index("candidate").loc["low_resolution_fast_unsafe", "statuses"]
gate_results, fast_unsafe_status


## 22. Global versus feasible Pareto frontiers

The unconstrained frontier is retained only as a teaching comparison. `global_pareto_efficient` means non-dominated in the declared optimization dimensions; it does **not** imply the candidate is safe or eligible. Operational selection uses `feasible_pareto_efficient`, computed only after every required Site-B gate passes. This prevents a zero-recall model from acquiring deployment significance merely because it is fast or small.


In [ ]:
def pareto_mask(frame, minimize, maximize):
    keep = []
    for i, row in frame.iterrows():
        dominated = False
        for j, other in frame.iterrows():
            if i == j: continue
            no_worse = all(other[col] <= row[col] for col in minimize) and all(other[col] >= row[col] for col in maximize)
            strictly_better = any(other[col] < row[col] for col in minimize) or any(other[col] > row[col] for col in maximize)
            if no_worse and strictly_better:
                dominated = True; break
        keep.append(not dominated)
    return np.asarray(keep)

PARETO_MINIMIZE = ("deterministic_service_p95_ms", "artifact_size_mb")
PARETO_MAXIMIZE = ("small_defect_recall", "retrieval_recall", "small_evidence_retention_rate")
candidate_matrix = candidate_matrix.merge(gate_results[["candidate", "eligible"]], on="candidate", how="left")
candidate_matrix["global_pareto_efficient"] = pareto_mask(candidate_matrix, PARETO_MINIMIZE, PARETO_MAXIMIZE)
candidate_matrix["feasible_pareto_efficient"] = False
feasible_index = candidate_matrix.index[candidate_matrix["eligible"]]
candidate_matrix.loc[feasible_index, "feasible_pareto_efficient"] = pareto_mask(
    candidate_matrix.loc[feasible_index], PARETO_MINIMIZE, PARETO_MAXIMIZE
)
assert not candidate_matrix.query("not eligible")["feasible_pareto_efficient"].any()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for axis, flag, title in (
    (axes[0], "global_pareto_efficient", "Unconstrained teaching frontier"),
    (axes[1], "feasible_pareto_efficient", "Operational frontier after hard gates"),
):
    for _, row in candidate_matrix.iterrows():
        on_front = bool(row[flag])
        axis.scatter(row["deterministic_service_p95_ms"], row["small_defect_recall"], s=90 if on_front else 40, marker="o" if on_front else "x")
        axis.annotate(row["candidate"], (row["deterministic_service_p95_ms"], row["small_defect_recall"]), fontsize=7)
    axis.set(xlabel="deterministic service p95 ms", title=title)
axes[0].set_ylabel("small-defect recall")
plt.tight_layout(); plt.show()
candidate_matrix[["candidate", "eligible", "global_pareto_efficient", "feasible_pareto_efficient"]]


## 23. Freeze Site-B policy, then open Site C once

Selection considers only feasible-Pareto optimized candidates and requires a material service-latency or artifact-size benefit. If no optimized candidate clears that objective, `KEEP_REFERENCE` is the correct decision—not a failed optimization. The complete policy—including thresholds, objective, cascade, cache, batching, and artifact lineage—is hashed before Site C.


In [ ]:
MINIMUM_LATENCY_IMPROVEMENT_FRACTION = 0.05
MINIMUM_ARTIFACT_REDUCTION_FRACTION = 0.25
reference_selection_row = candidate_matrix.set_index("candidate").loc["reference"]
candidate_matrix["latency_improvement_fraction"] = 1 - (
    candidate_matrix["deterministic_service_p95_ms"] / reference_selection_row["deterministic_service_p95_ms"]
)
candidate_matrix["artifact_reduction_fraction"] = 1 - (
    candidate_matrix["artifact_size_mb"] / reference_selection_row["artifact_size_mb"]
)
candidate_matrix["meaningful_systems_benefit"] = (
    (candidate_matrix["latency_improvement_fraction"] >= MINIMUM_LATENCY_IMPROVEMENT_FRACTION)
    | (candidate_matrix["artifact_reduction_fraction"] >= MINIMUM_ARTIFACT_REDUCTION_FRACTION)
)

def select_optimized_or_reference(frame):
    pool = frame[
        (frame["candidate"] != "reference")
        & frame["eligible"]
        & frame["feasible_pareto_efficient"]
        & frame["meaningful_systems_benefit"]
    ]
    if pool.empty:
        return {"candidate": "reference", "selection_outcome": "KEEP_REFERENCE"}
    winner = pool.sort_values(["deterministic_service_p95_ms", "artifact_size_mb"]).iloc[0]
    return {"candidate": str(winner["candidate"]), "selection_outcome": "PROMOTE_OPTIMIZED"}

site_b_selection = select_optimized_or_reference(candidate_matrix)
selected_name = site_b_selection["candidate"]
no_benefit_demo = candidate_matrix.copy()
no_benefit_demo.loc[no_benefit_demo["candidate"] != "reference", "meaningful_systems_benefit"] = False
keep_reference_demo = select_optimized_or_reference(no_benefit_demo)
assert keep_reference_demo["selection_outcome"] == "KEEP_REFERENCE"
MODEL_REGISTRY = {
    "reference": reference_model, "int8_like": quantized_model,
    "int8_same_accuracy_bad_calibration": quantized_model,
    "structured_width_6": pruned_models[6], "task_only_student": task_only_student,
    "multimodal_student": multimodal_student, "token_pruning_50": reference_model,
    "low_resolution_fast_unsafe": reference_model,
}
RESOLUTION_REGISTRY = {"low_resolution_fast_unsafe": 8}
LOGIT_SCALE_REGISTRY = {"int8_same_accuracy_bad_calibration": 0.15}

selected_model = MODEL_REGISTRY[selected_name]
selected_row = candidate_matrix.set_index("candidate").loc[selected_name]
optimization_policy = {
    "selected_candidate": selected_name,
    "site_b_selection_outcome": site_b_selection["selection_outcome"],
    "minimum_latency_improvement_fraction": MINIMUM_LATENCY_IMPROVEMENT_FRACTION,
    "minimum_artifact_reduction_fraction": MINIMUM_ARTIFACT_REDUCTION_FRACTION,
    "deployment_budget": DEPLOYMENT_BUDGET,
    "defect_threshold": DEFECT_THRESHOLD,
    "cascade_threshold": CASCADE_THRESHOLD,
    "batch_policy": {"max_batch": 4, "wait_ms": 3.0, "queue_limit": 16, "timeout_ms": 40.0},
    "cache_policy": "content+encoder+processor+tenant+principal+scope/v2",
    "source_roles": [asdict(source) for source in SOURCES],
    "selection_source": "Site B only",
}
POLICY_HASH_BEFORE_SITE_C = canonical_hash(optimization_policy)

# Site C is evaluated only after the full policy is frozen.
site_c_capability = evaluate_capabilities(
    selected_model, site_c,
    resolution=RESOLUTION_REGISTRY.get(selected_name, 24),
    logit_scale=LOGIT_SCALE_REGISTRY.get(selected_name, 1.0),
)
site_c_host_timing = benchmark_forward(selected_model, resize_images(site_c["x"][:1], RESOLUTION_REGISTRY.get(selected_name, 24)), 30)
site_c_service_model = deterministic_service_latency(
    selected_model,
    resolution=RESOLUTION_REGISTRY.get(selected_name, 24),
    bytes_per_parameter=1 if selected_name.startswith("int8") else 4,
    token_retention=0.5 if selected_name == "token_pruning_50" else 1.0,
    workload_multiplier=1.10,
)
site_c_queue_model = load_results.query("arrival_rate_rps == 900 and wait_ms == 3.0").iloc[0]
site_c_system = {
    **site_c_host_timing,
    **site_c_service_model,
    "deterministic_end_to_end_p95_ms": float(site_c_queue_model["end_to_end_p95_ms"]),
    "cold_start_ms": system_baseline["cold_start_ms"],
    "throughput_requests_per_s": float(batch_results["throughput_requests_per_s"].max()),
    "cache_valid_hit_rate": 0.72,
    "queue_timeout_rate_at_shifted_load": float(site_c_queue_model["timeout_or_reject_rate"]),
}
POLICY_HASH_AFTER_SITE_C = canonical_hash(optimization_policy)
assert POLICY_HASH_BEFORE_SITE_C == POLICY_HASH_AFTER_SITE_C
pd.DataFrame([{"candidate": selected_name, "policy_hash": POLICY_HASH_BEFORE_SITE_C, **site_c_system, **site_c_capability}])


## 24. Composition failure and ordered transformation lineage

Individually acceptable transformations are not assumed independent. The explicit combined candidate composes resolution reduction, quantization, and the student, then reruns the gate.


In [ ]:
combined_model = ptq_proxy(multimodal_student)
combined_row = candidate_row("combined_student_int8_lowres", combined_model, resolution=12, bytes_per_parameter=1)
combined_checks = evaluate_gate(pd.Series(combined_row))
composition_result = {
    "candidate": combined_row["candidate"],
    "ordered_transformations": ["multimodal_distillation", "int8_like_ptq", "resolution_12"],
    "parent_digest": REFERENCE_DIGEST,
    "candidate_digest": state_digest(combined_model),
    "checks": {check.name: check.status for check in combined_checks},
    "eligible": all(check.status == "PASS" for check in combined_checks),
}
composition_result


## 25. Governed evidence and non-authorizing deployment decision

The trusted gate consumes deterministic service/load evidence, capability measurements, and exact lineage. Its terminal state is one of `PROMOTE_OPTIMIZED`, `KEEP_REFERENCE`, `REJECT`, or `MISSING_EVIDENCE`; none grants production authorization. Rollback remains the immutable reference.


In [ ]:
site_c_gate_row = selected_row.copy()
site_c_gate_row["deterministic_service_p95_ms"] = site_c_system["deterministic_service_p95_ms"]
for key, value in site_c_capability.items():
    site_c_gate_row[key] = value
selected_checks = evaluate_gate(site_c_gate_row) + (
    GateCheck(
        "end_to_end_tail_latency",
        "PASS" if site_c_system["deterministic_end_to_end_p95_ms"] <= DEPLOYMENT_BUDGET["deterministic_end_to_end_p95_ms_max"] else "FAIL",
        f"observed={site_c_system['deterministic_end_to_end_p95_ms']:.6g}; max={DEPLOYMENT_BUDGET['deterministic_end_to_end_p95_ms_max']:.6g}",
    ),
    GateCheck(
        "queue_timeout_rate",
        "PASS" if site_c_system["queue_timeout_rate_at_shifted_load"] <= DEPLOYMENT_BUDGET["queue_timeout_rate_max"] else "FAIL",
        f"observed={site_c_system['queue_timeout_rate_at_shifted_load']:.6g}; max={DEPLOYMENT_BUDGET['queue_timeout_rate_max']:.6g}",
    ),
)
statuses = {check.status for check in selected_checks}
if "MISSING" in statuses:
    outcome = "MISSING_EVIDENCE"
elif "FAIL" in statuses:
    outcome = "REJECT"
elif selected_name == "reference":
    outcome = "KEEP_REFERENCE"
else:
    outcome = "PROMOTE_OPTIMIZED"
decision = DeploymentDecision(selected_name, outcome, selected_checks, POLICY_HASH_BEFORE_SITE_C)
assert decision.authorization == "none"

artifact_lineage = {
    "source_model": {"artifact_id": "reference-fp32", "digest": REFERENCE_DIGEST},
    "selected_candidate": {"artifact_id": selected_name, "digest": state_digest(selected_model)},
    "ordered_transformations": [selected_name],
    "runtime": "pytorch_eager_cpu_teaching_proxy",
    "hardware_compatibility": environment_manifest,
    "rollback_artifact": "reference-fp32",
}
evidence_bundle = {
    "course": "Advanced 08 — Efficient Spatial & Multimodal Inference",
    "generated_at_policy_date": "2026-09-20",
    "teaching_measurements_only": True,
    "environment": environment_manifest,
    "workload": asdict(WORKLOAD),
    "sources": [asdict(source) for source in SOURCES],
    "reference_digest": REFERENCE_DIGEST,
    "site_b_policy": optimization_policy,
    "site_b_selection": site_b_selection,
    "keep_reference_counterexample": keep_reference_demo,
    "pareto_semantics": {
        "global_pareto_efficient": "teaching-only unconstrained non-domination",
        "feasible_pareto_efficient": "non-domination after every required gate passes",
    },
    "timing_method": {
        "host_measurement_role": "diagnostic only",
        "stage_profile_repetitions": WORKLOAD.measured_runs,
        "candidate_host_repetitions": 20,
        "site_c_host_repetitions": 30,
        "host_inner_loops_per_timed_region": WORKLOAD.timed_inner_loops,
        "release_gate_source": "deterministic_workload_and_queue_model_v1",
    },
    "behavioral_parity_counterexample": behavioral_parity_counterexample,
    "policy_hash": POLICY_HASH_BEFORE_SITE_C,
    "policy_hash_unchanged_after_site_c": POLICY_HASH_BEFORE_SITE_C == POLICY_HASH_AFTER_SITE_C,
    "selected_candidate": selected_name,
    "site_c_system_metrics": site_c_system,
    "site_c_capability_metrics": site_c_capability,
    "cache_attack": cache_attack_results.to_dict(orient="records"),
    "composition": composition_result,
    "artifact_lineage": artifact_lineage,
    "decision": asdict(decision),
    "authorization": "none",
}
evidence_path = ARTIFACT_DIR / "efficient_inference_evidence.json"
decision_path = ARTIFACT_DIR / "efficient_inference_candidates.csv"
evidence_path.write_text(json.dumps(evidence_bundle, indent=2, default=str) + "\n", encoding="utf-8")
candidate_matrix.merge(gate_results[["candidate", "statuses"]], on="candidate", how="left").to_csv(decision_path, index=False)
loaded = json.loads(evidence_path.read_text(encoding="utf-8"))
assert loaded["authorization"] == "none"
assert loaded["policy_hash_unchanged_after_site_c"] is True
assert loaded["artifact_lineage"]["rollback_artifact"] == "reference-fp32"
asdict(decision), evidence_path, decision_path


## 26. Failure attribution

Use the earliest observable cause rather than calling every regression “accuracy loss.”


In [ ]:
failure_taxonomy = pd.DataFrame([
    ("Input Budget Failure", "small defect vanishes after resize", "resolution sweep"),
    ("Token-Pruning Failure", "critical token removed", "evidence-retention fixture"),
    ("Quantization Failure", "parity or calibration regresses", "PTQ comparison"),
    ("Pruning Failure", "narrow tensor loses capability", "structured curve"),
    ("Distillation Failure", "task preserved; retrieval lost", "capability suite"),
    ("Compilation Parity Failure", "runtime disagrees", "numeric + task parity"),
    ("Cache Invalidation Failure", "stale or cross-scope hit", "cache assertions"),
    ("Queueing Failure", "tail/deadline/timeout collapse", "load test"),
    ("Memory-Budget Failure", "peak exceeds target", "memory evidence"),
    ("Reliability Regression", "ECE/OOD policy changes", "Advanced 07 suite"),
    ("Unsupported Runtime", "operator/backend unavailable", "compatibility manifest"),
], columns=["error_type", "example", "evidence"])
failure_taxonomy


## 27. Production checklist

Before an actual release:

- reproduce on target hardware with synchronized device timing, process memory, power, and sustained thermal runs;
- benchmark real decode, preprocessing, network, queue, postprocessing, and policy stages;
- use licensed, source/time/site-isolated data and representative quantization calibration slices;
- evaluate every claimed capability, reliability threshold, critical slice, and degraded mode;
- inspect compiler coverage, unsupported operators, dynamic-shape profiles, numeric and task parity;
- bind model, processor, cache, runtime, policy, hardware, and rollback revisions;
- test cancellation, saturation, deadline, freshness, network loss, cache purge, and rollback;
- protect embeddings, tokens, and temporal state with authorization, tenant isolation, encryption, and retention; and
- require trusted release approval. This notebook grants none.


## 28. Exercises

1. Add a calibrated early-exit head and compare it with permanent layer dropping.
2. Replace per-tensor PTQ with a per-channel proxy and inspect outlier sensitivity.
3. Add token merging and compare critical-evidence retention at equal token counts.
4. Define bounded dynamic shapes in `torch.export` and test every boundary shape.
5. Add retrieval-cache invalidation after an encoder revision.
6. Repeat distillation and pruning across seeds and report variability.
7. Add direct process RSS or device memory on a controlled target.
8. Replace the queue simulator with a real inference server load test while preserving the same evidence schema.


## 29. Explain without code

- Why can fewer FLOPs be slower?
- Why can average accuracy survive while critical evidence disappears?
- Why is sparsity not speed?
- Why can quantization require a new calibration and OOD policy?
- Why is task-only distillation unsafe for a retrieval-capable system?
- Why can dynamic batching improve throughput and worsen p95?
- Why is a cache hit not automatically valid?
- Why can a Pareto-efficient candidate still fail deployment constraints?
- Why must Site C remain reporting-only?
- Why does the release service—not the optimizer—own promotion and rollback?
